# 06 — Full universe and final report

**Phase 6 deliverable:** a comparison report across models × agents × tickers, with the naive
baseline pinned to the top of every table.

The ticker order is deliberate — **KBANK (clean) → SCB (structural break) → BAY (thin float)**.
Each adds one distinct failure mode, and you want them isolated rather than averaged together.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

## The three tickers are not equivalent instruments

- **KBANK** — no known discontinuity, liquid. The clean case, and the one to bring end-to-end first.
- **SCB** — the ticker survived the April 2022 SCBX restructuring but the issuer changed from a
  bank to a holding company. ~4 years of usable history under the default policy.
- **BAY** — ~72–76% MUFG-held. Thin float, so a backtest that assumes it can transact at the close
  in any size is assuming liquidity that is not there. Carries a 5% participation cap by default.

In [ ]:
from stock_retrofit.data import describe, read_meta

for symbol in ["KBANK", "SCB", "BAY"]:
    meta = read_meta(symbol)
    print(describe(symbol))
    if meta:
        print(f"    {meta.rows} bars {meta.start} -> {meta.end}, hash {meta.content_hash[:12]}")
    print()

## Generate the report

Equivalent to:

```bash
python -m stock_retrofit.cli report --symbols KBANK,SCB,BAY
```

Results already in `results/` are reused; anything missing is computed. Writes
`results/final-report.md`.

In [ ]:
from stock_retrofit.report import build_report
from stock_retrofit.paths import RESULTS_DIR
from IPython.display import Markdown

text = build_report(["KBANK", "SCB", "BAY"], run_missing=True)
(RESULTS_DIR / "final-report.md").write_text(text)
Markdown(text)

## The headline, and why it is not a failure

Acceptance criterion 9 requires the report to state plainly how many models beat `NaiveLag`
out-of-sample after costs — **including if the answer is zero.**

If it is zero, that is the finding. The upstream repository reports accuracies in the high
nineties for these same architectures, and both things are true at once. The difference is
methodological, not architectural:

| | upstream | here |
|---|---|---|
| scaler | fit on the full series, then split | fit inside each fold |
| split | one fixed 30-day tail | walk-forward, 8 folds |
| target | price levels | next-day returns |
| baseline | none | `NaiveLag` on every table |
| agent evaluation | in-sample (`get_reward` and `buy` share `self.trend`) | held-out folds only |
| trading costs | none | board lot, ticks, commission + VAT, ±30% limits |

A naive lag scores in the high nineties on upstream's metric too. Those numbers never measured
skill, so failing to reproduce them is the correct outcome.

## Provenance

Every run in `results/` has a `*.manifest.json` beside it recording the config, the git SHA and
the content hash of the bars consumed.

In [ ]:
import json
from stock_retrofit.paths import RESULTS_DIR

manifests = sorted(RESULTS_DIR.glob("*.manifest.json"))
print(f"{len(manifests)} manifests\n")
if manifests:
    payload = json.loads(manifests[-1].read_text())
    print(json.dumps({k: payload[k] for k in ["run_id", "created_at", "git_sha", "seed", "data"]},
                     indent=2))